In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 1. Giriş

Bu projede Titanic veri seti kullanılmıştır. Bu veri seti, 1912 yılında yaşanan Titanic gemi kazasında yer alan yolcuların bilgilerini içermektedir.
Amaç, yolcuların belirli özelliklerine (yaş, cinsiyet, bilet sınıfı, ücret vb.) bakarak
hayatta kalıp kalmadıklarını tahmin edebilen bir model geliştirmektir.

Veri seti, Kaggle tarafından sağlanan “Titanic: Machine Learning from Disaster” adlı klasik bir makine öğrenmesi yarışmasına aittir.
Toplamda 891 yolcuya ait veri bulunmaktadır.

| Değişken | Açıklama |
|-----------|-----------|
| PassengerId | Yolcu kimlik numarası |
| Survived | Hayatta kalma durumu (0 = Hayır, 1 = Evet) |
| Pclass | Yolcu sınıfı |
| Name | Yolcunun tam adı |
| Sex | Cinsiyet (male / female) |
| Age | Yaş |
| SibSp | Gemideki kardeş/eş sayısı |
| Parch | Gemideki ebeveyn/çocuk sayısı |
| Ticket | Bilet numarası |
| Fare | Bilet ücreti |
| Cabin | Kabin numarası |
| Embarked | Biniş limanı |


## 2. Kütüphanelerin Yüklenmesi

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

## 3. Verinin Yüklenmesi

In [ ]:
train = pd.read_csv("/kaggle/input/titanic/train.csv")

## 4. Keşifsel Veri Analizi

In [ ]:
train.head()

In [ ]:
train.describe()

In [ ]:
train.isnull().sum()

In [ ]:
sns.barplot(x = "Sex", y = "Survived", data = train)
plt.title("Cinsiyete gore hayatta kalma")
plt.show()

In [ ]:
sns.barplot(x = "Pclass", y = "Survived", data = train)
plt.title("Yolcu sınıfına göre hayatta kalma")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data=train, x="Age", hue="Survived", multiple="stack", kde=True)
plt.title("Yaşa göre hayatta kalma")
plt.show()

## 5. Feature Engineering

In [ ]:
train['Title'] = train['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
train['Title'] = train['Title'].replace(['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
train['Title'] = train['Title'].replace(['Mlle','Ms'], 'Miss')
train['Title'] = train['Title'].replace('Mme', 'Mrs')

In [ ]:
sns.barplot(x = "Title", y = "Survived", data = train)
plt.title("Title'a Göre Hayatta Kalma")
plt.show()

In [ ]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1

In [ ]:
sns.barplot(x = "FamilySize", y = "Survived", data = train)
plt.title("Aile Boyutuna Göre Hayatta Kalma Oranı")
plt.show()

In [ ]:
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)

In [ ]:
sns.barplot(x = "IsAlone", y = "Survived", data = train)
plt.title("Yolcunun Yalnız Olma Durumuna Göre Hayatta Kalma Oranı")
plt.show()

In [ ]:
train['CabinDeck'] = train['Cabin'].astype(str).str[0]
train['CabinDeck'] = train['CabinDeck'].replace('n', 'Unknown')

In [ ]:
sns.barplot(x = "CabinDeck", y = "Survived", data = train,order=sorted(train["CabinDeck"].unique()))
plt.title("Güverteye Göre Hayatta Kalma Oranı")
plt.show()

In [ ]:
train['TicketCount'] = train.groupby('Ticket')['Ticket'].transform('count')

In [ ]:
sns.barplot(x = "TicketCount", y = "Survived", data = train)
plt.title("Bilet Paylaşan Kişi Sayısına Göre Hayatta Kalma Oranı")
plt.show()

In [ ]:
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})

## 6. Eğitim Verisinin Hazırlanması

In [ ]:
y = train['Survived']
train.drop(['Survived','Name','Ticket','Cabin','PassengerId'], axis=1, inplace=True)

In [ ]:
sns.heatmap(train.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Feature Correlation Heatmap')

Modelde Parch ve TicketCount değişkenleri kullanılmamıştır.
Bu iki değişkenin FamilySize gibi diğer değişkenlerle yüksek korelasyon göstermesi nedeniyle modelden çıkarılmıştır.
Bu sayede gereksiz tekrarın önüne geçilerek modelin daha dengeli öğrenmesi hedeflenmiştir.

## 7. Eksik Verileri Doldurma

In [ ]:
numeric_features = ['Age', 'Fare','FamilySize','IsAlone']
categorical_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'CabinDeck']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
])

## 8. Verinin Bölünmesi

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_val.shape)

## 9. XGBoost Model

In [ ]:
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    ))
])

xgb_model.fit(X_train, y_train)
xgb_val_preds = xgb_model.predict(X_val)
xgb_val_acc = accuracy_score(y_val, xgb_val_preds)
print(f"XGBoost validation accuracy: {xgb_val_acc:.4f}")

Bu çalışmada Titanic veri seti üzerinde temel bir veri analizi ve sınıflandırma modeli uyguladım.
Veri ön işleme, feature engineering ve modelleme adımlarını deneyerek süreci daha iyi anlamayı hedefledim.
Modelin performansını artırmak için farklı değişken kombinasyonlarını test ettim ve gereksiz, yüksek korelasyonlu değişkenleri çıkardım.

Bu çalışmayı oluştururken, Kaggle üzerinde yer alan Muhammad Riaz – “Titanic Survival Prediction (Clean EDA + Machine Learning)” adlı notebook’tan da fikir aldım ve kendi yorumlarımla uyarladım.
Bu süreç sayesinde veri analizi ve modelleme aşamalarının mantığını daha iyi kavradım.